# RAG + MNIST — Final Assignment


This notebook contains the two final tasks from the course notebook:

1. `retriver` class
   - `chunker()`
   - `embedder()`
   - `search()`
2. `network` class
   - 4-layer `Sequential` neural network
   - custom `train()`
   - custom `predict()`
   - MNIST handwritten-digit classification



In [20]:
import sys
import platform

print("Python:", sys.version)
print("Executable:", sys.executable)
print("Platform:", platform.platform())

if sys.version_info[:2] != (3, 10):
    print("\nWARNING: This notebook is intended for Python 3.10.")
else:
    print("\n✓ Python 3.10 detected.")

Python: 3.10.20 (main, Jun 23 2026, 15:19:56) [MSC v.1944 64 bit (AMD64)]
Executable: E:\AI-ML\.venv310\Scripts\python.exe
Platform: Windows-10-10.0.26200-SP0

✓ Python 3.10 detected.


## Part 1 — Retriever

The goal is simple: **document → chunks → embeddings → similarity search**.

In [21]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [22]:
class retriver:
    """
    A simple semantic retriever.

    Required assignment methods:
        chunker()  : document -> chunks
        embedder() : chunks -> embeddings
        search()   : query -> most similar chunks
    """

    def __init__(
        self,
        document,
        chunk_size=500,
        overlap=50,
        model_name="all-MiniLM-L6-v2"
    ):
        if not isinstance(document, str):
            raise TypeError("document must be a string.")

        if not document.strip():
            raise ValueError("document cannot be empty.")

        if chunk_size <= 0:
            raise ValueError("chunk_size must be greater than 0.")

        if overlap < 0 or overlap >= chunk_size:
            raise ValueError(
                "overlap must be >= 0 and smaller than chunk_size."
            )

        self.document = document.strip()
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.chunks = []
        self.embeddings = None

        self.model = SentenceTransformer(model_name)

    def chunker(self):
        """Split the document into overlapping character chunks."""

        self.chunks = []
        start = 0
        n = len(self.document)

        while start < n:
            end = min(start + self.chunk_size, n)
            chunk = self.document[start:end].strip()

            if chunk:
                self.chunks.append(chunk)

            if end == n:
                break

            start = end - self.overlap

        return self.chunks

    def embedder(self):
        """Convert chunks into normalized dense vectors."""

        if not self.chunks:
            self.chunker()

        self.embeddings = self.model.encode(
            self.chunks,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False
        )

        return self.embeddings

    def search(self, query, top_k=3):
        """Return the top-k chunks ranked by cosine similarity."""

        if not isinstance(query, str):
            raise TypeError("query must be a string.")

        if not query.strip():
            raise ValueError("query cannot be empty.")

        if top_k <= 0:
            raise ValueError("top_k must be greater than 0.")

        if self.embeddings is None:
            self.embedder()

        query_embedding = self.model.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False
        )

        scores = cosine_similarity(
            query_embedding,
            self.embeddings
        )[0]

        top_k = min(top_k, len(self.chunks))
        indices = np.argsort(scores)[-top_k:][::-1]

        return [
            {
                "rank": rank,
                "chunk_index": int(index),
                "score": float(scores[index]),
                "text": self.chunks[index]
            }
            for rank, index in enumerate(indices, start=1)
        ]

    def show_results(self, query, top_k=3):
        """Pretty-print search results."""

        results = self.search(query, top_k)

        print("=" * 80)
        print("QUERY:", query)
        print("=" * 80)

        for item in results:
            print(f"\nRank       : {item['rank']}")
            print(f"Chunk      : {item['chunk_index']}")
            print(f"Similarity : {item['score']:.4f}")
            print("Text       :", item["text"])
            print("-" * 80)

In [23]:
document = """
Machine learning is a branch of artificial intelligence that enables
computers to learn patterns from data without being explicitly programmed
for every individual task.

Supervised learning uses labeled data. Classification and regression are
common examples of supervised learning.

Unsupervised learning uses unlabeled data to discover hidden structures,
patterns, and relationships. Clustering is a common example.

Deep learning is a subfield of machine learning that uses artificial
neural networks with multiple layers. It is widely used for image
recognition, speech recognition, and natural language processing.

Natural language processing, or NLP, enables computers to process and
understand human language.

Computer vision enables computers to interpret images and videos.

Reinforcement learning allows an agent to learn by interacting with an
environment and receiving rewards or penalties.
"""

retriever = retriver(
    document=document,
    chunk_size=300,
    overlap=50
)

chunks = retriever.chunker()

print("Number of chunks:", len(chunks))
for i, chunk in enumerate(chunks):
    print(f"\n--- CHUNK {i} ---")
    print(chunk)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Number of chunks: 4

--- CHUNK 0 ---
Machine learning is a branch of artificial intelligence that enables
computers to learn patterns from data without being explicitly programmed
for every individual task.

Supervised learning uses labeled data. Classification and regression are
common examples of supervised learning.

Unsupervised le

--- CHUNK 1 ---
examples of supervised learning.

Unsupervised learning uses unlabeled data to discover hidden structures,
patterns, and relationships. Clustering is a common example.

Deep learning is a subfield of machine learning that uses artificial
neural networks with multiple layers. It is widely used for i

--- CHUNK 2 ---
orks with multiple layers. It is widely used for image
recognition, speech recognition, and natural language processing.

Natural language processing, or NLP, enables computers to process and
understand human language.

Computer vision enables computers to interpret images and videos.

Reinforcement

--- CHUNK 3 ---
ers to int

In [24]:
embeddings = retriever.embedder()

print("Number of chunks:", len(retriever.chunks))
print("Embedding matrix shape:", embeddings.shape)

Number of chunks: 4
Embedding matrix shape: (4, 384)


In [25]:
retriever.show_results(
    "What is deep learning?",
    top_k=3
)

QUERY: What is deep learning?

Rank       : 1
Chunk      : 1
Similarity : 0.5624
Text       : examples of supervised learning.

Unsupervised learning uses unlabeled data to discover hidden structures,
patterns, and relationships. Clustering is a common example.

Deep learning is a subfield of machine learning that uses artificial
neural networks with multiple layers. It is widely used for i
--------------------------------------------------------------------------------

Rank       : 2
Chunk      : 0
Similarity : 0.5049
Text       : Machine learning is a branch of artificial intelligence that enables
computers to learn patterns from data without being explicitly programmed
for every individual task.

Supervised learning uses labeled data. Classification and regression are
common examples of supervised learning.

Unsupervised le
--------------------------------------------------------------------------------

Rank       : 3
Chunk      : 3
Similarity : 0.3980
Text       : ers to interpre

### Retriever flow

```text
Document
   ↓
chunker()
   ↓
Text chunks
   ↓
embedder()
   ↓
Vector embeddings
   ↓
search(query)
   ↓
Most similar chunks
```

The important idea is that the query and document chunks are represented in the same vector space. Cosine similarity is then used to rank the chunks.

## Part 2 — MNIST Neural Network

The assignment requires a `network` class, custom training/prediction methods, four layers inside `Sequential`, and MNIST.

In [26]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras import layers
from tensorflow.keras.datasets import mnist

print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


In [27]:
class network:
    """
    Four-layer Sequential neural network for MNIST.

    Layers:
        1. Flatten
        2. Dense(128, ReLU)
        3. Dense(64, ReLU)
        4. Dense(10, Softmax)

    Custom methods:
        train()
        predict()
    """

    def __init__(self):

        self.model = Sequential([
            layers.Flatten(input_shape=(28, 28)),
            layers.Dense(128, activation="relu"),
            layers.Dense(64, activation="relu"),
            layers.Dense(10, activation="softmax")
        ])

        self.model.compile(
            optimizer="adam",
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )

    def train(
        self,
        x_train,
        y_train,
        epochs=5,
        batch_size=32,
        validation_split=0.1
    ):
        """Train the network and return the Keras History object."""

        return self.model.fit(
            x_train,
            y_train,
            epochs=epochs,
            batch_size=batch_size,
            validation_split=validation_split,
            verbose=1
        )

    def predict(self, x):
        """Return the predicted digit for each input image."""

        probabilities = self.model.predict(
            x,
            verbose=0
        )

        return np.argmax(probabilities, axis=1)

    def evaluate(self, x_test, y_test):
        """Evaluate the trained model on test data."""

        return self.model.evaluate(
            x_test,
            y_test,
            verbose=0
        )

    def summary(self):
        """Display the four-layer architecture."""

        return self.model.summary()

In [28]:
# Load MNIST
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize pixel values from [0, 255] to [0, 1]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

print("Training images:", x_train.shape)
print("Training labels:", y_train.shape)
print("Testing images :", x_test.shape)
print("Testing labels :", y_test.shape)

Training images: (60000, 28, 28)
Training labels: (60000,)
Testing images : (10000, 28, 28)
Testing labels : (10000,)


In [29]:
net = network()
net.summary()

E:\AI-ML\.venv310\lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ flatten_2 (Flatten)                  │ (None, 784)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 128)                 │         100,480 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_8 (Dense)                      │ (None, 10)                  │             650 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 109,386 (427.29 KB)

 Trainable params: 109,386 (427.29 KB)

 Non-trainable params: 0 (0.00 B)

In [30]:
history = net.train(
    x_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.1
)

Epoch 1/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 24s 11ms/step - accuracy: 0.9260 - loss: 0.2543 - val_accuracy: 0.9688 - val_loss: 0.1088
Epoch 2/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - accuracy: 0.9666 - loss: 0.1080 - val_accuracy: 0.9730 - val_loss: 0.0900
Epoch 3/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 14s 8ms/step - accuracy: 0.9767 - loss: 0.0756 - val_accuracy: 0.9767 - val_loss: 0.0798
Epoch 4/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 14s 8ms/step - accuracy: 0.9822 - loss: 0.0570 - val_accuracy: 0.9803 - val_loss: 0.0734
Epoch 5/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 14s 8ms/step - accuracy: 0.9857 - loss: 0.0449 - val_accuracy: 0.9780 - val_loss: 0.0879


In [31]:
test_loss, test_accuracy = net.evaluate(
    x_test,
    y_test
)

print(f"Test loss     : {test_loss:.4f}")
print(f"Test accuracy : {test_accuracy:.4f}")

Test loss     : 0.0837
Test accuracy : 0.9748


In [32]:
predictions = net.predict(x_test[:10])

print("Predicted:", predictions)
print("Actual   :", y_test[:10])

Predicted: [7 2 1 0 4 1 4 9 6 9]
Actual   : [7 2 1 0 4 1 4 9 5 9]


# Final checklist

## Retriever assignment
- [x] `retriver` class
- [x] `chunker()`
- [x] `embedder()`
- [x] `search()`
- [x] cosine similarity
- [x] ranked top-k results

## Neural-network assignment
- [x] `network` class
- [x] custom `train()`
- [x] custom `predict()`
- [x] 4 layers
- [x] `Sequential`
- [x] MNIST
- [x] normalization
- [x] evaluation
- [x] prediction

